# GoldenDentist — Train cephalometric landmark detector on Aariz

**Self-contained Colab notebook.** Run cells top to bottom.

1. *Runtime → Change runtime type → T4 GPU* (free).
2. Upload the **Aariz** dataset folder to your Google Drive once.
3. Run all cells. The trained `ceph.onnx` is downloaded to your machine at the end.

No uploads, no git clone. The notebook writes every Python source file itself.

## 1. Install dependencies

In [ ]:
%pip install --quiet torch torchvision albumentations opencv-python tqdm onnx onnxruntime

## 2. Confirm GPU is attached and prepare workspace

In [ ]:
import os, torch
os.makedirs('/content/ai', exist_ok=True)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
else:
    print('NO GPU — go to Runtime > Change runtime type > T4 GPU.')


## 3. Write every project Python file into `/content/ai/`

Each cell uses Jupyter's `%%writefile` magic to materialize one module. Run them in order.

In [ ]:
%%writefile /content/ai/landmarks_mapping.py
"""Mapping between Aariz / CEPHA29 annotations and the 19 landmarks
GoldenDentist uses in its UI.

Aariz JSON files store landmarks as an *ordered* array; each entry
carries a `symbol` field (e.g. "S", "ANS", "UIT"). We look up by
symbol — never by array index — so the mapping is robust even if a
future Aariz release reorders the array.

Verified against the actual dataset on 2026-04-30. The 29 Aariz
symbols, in the order they appear in train/.../Senior Orthodontists/*.json,
are:

      0  A      A-point
      1  ANS    Anterior Nasal Spine
      2  B      B-point
      3  Me     Menton
      4  N      Nasion
      5  Or     Orbitale
      6  Pog    Pogonion
      7  PNS    Posterior Nasal Spine
      8  Pn     Pronasale
      9  R      Ramus
     10  S      Sella
     11  Ar     Articulare
     12  Co     Condylion
     13  Gn     Gnathion
     14  Go     Gonion
     15  Po     Porion
     16  LPM    Lower 2nd PM Cusp Tip
     17  LIT    Lower Incisor Tip
     18  LMT    Lower Molar Cusp Tip
     19  UPM    Upper 2nd PM Cusp Tip
     20  UIA    Upper Incisor Apex
     21  UIT    Upper Incisor Tip
     22  UMT    Upper Molar Cusp Tip
     23  LIA    Lower Incisor Apex
     24  Li     Labrale inferius
     25  Ls     Labrale superius
     26  N`     Soft Tissue Nasion
     27  Pog`   Soft Tissue Pogonion
     28  Sn     Subnasale
"""
from __future__ import annotations

from typing import Iterable

import numpy as np


# Order produced by the model — MUST match
# js/ai/onnx.js::LANDMARK_ORDER and ai/model.py::LANDMARK_ORDER.
OUR_LANDMARK_ORDER = [
    "S", "N", "Or", "Po", "Ar",
    "ANS", "PNS", "A",
    "B", "Pog", "Gn", "Me", "Go",
    "U1", "U1A", "L1", "L1A",
    "OccA", "OccP",
]
NUM_OUR_LANDMARKS = len(OUR_LANDMARK_ORDER)  # 19


# Map each of our 19 landmarks to the Aariz symbol that matches it.
# `None` means we derive the value from other landmarks (see below).
OUR_TO_AARIZ_SYMBOL: dict[str, str | None] = {
    "S":   "S",
    "N":   "N",
    "Or":  "Or",
    "Po":  "Po",
    "Ar":  "Ar",
    "ANS": "ANS",
    "PNS": "PNS",
    "A":   "A",
    "B":   "B",
    "Pog": "Pog",
    "Gn":  "Gn",
    "Me":  "Me",
    "Go":  "Go",
    "U1":  "UIT",   # Upper Incisor Tip
    "U1A": "UIA",   # Upper Incisor Apex
    "L1":  "LIT",   # Lower Incisor Tip
    "L1A": "LIA",   # Lower Incisor Apex
    "OccA": None,   # derived: mid(U1, L1) — incisor occlusion midpoint
    "OccP": None,   # derived: mid(UMT, LMT) — molar occlusion midpoint
}


def aariz_dict(landmarks: Iterable[dict]) -> dict[str, np.ndarray]:
    """Convert the ``landmarks`` list from an Aariz JSON to a
    {symbol: np.array([x, y])} dict for easy lookup.
    """
    out: dict[str, np.ndarray] = {}
    for lm in landmarks:
        sym = lm["symbol"]
        v = lm["value"]
        out[sym] = np.array([float(v["x"]), float(v["y"])], dtype=np.float32)
    return out


def derive_occlusal(
    coords19: np.ndarray, aariz_by_symbol: dict[str, np.ndarray]
) -> np.ndarray:
    """Fill OccA / OccP. Aariz does not annotate the functional-occlusal-plane
    midpoints directly, but we have everything we need:

    * OccA  = midpoint(U1, L1)         — incisor occlusion midpoint.
    * OccP  = midpoint(UMT, LMT)       — molar occlusion midpoint
                                          (uses Aariz's UMT/LMT).

    If UMT or LMT is missing for some reason we fall back to projecting
    OccA onto the Go-Me line.
    """
    idx = {n: i for i, n in enumerate(OUR_LANDMARK_ORDER)}
    out = coords19.copy()

    U1, L1 = out[idx["U1"]], out[idx["L1"]]
    out[idx["OccA"]] = 0.5 * (U1 + L1)

    UMT = aariz_by_symbol.get("UMT")
    LMT = aariz_by_symbol.get("LMT")
    if UMT is not None and LMT is not None:
        out[idx["OccP"]] = 0.5 * (UMT + LMT)
    else:
        Go, Me = out[idx["Go"]], out[idx["Me"]]
        d = Me - Go
        norm = float(np.linalg.norm(d)) + 1e-9
        u = d / norm
        t = float(np.dot(out[idx["OccA"]] - Go, u))
        foot = Go + t * u
        out[idx["OccP"]] = 0.6 * Go + 0.4 * foot
    return out


def map_aariz_to_ours(landmarks: Iterable[dict]) -> np.ndarray:
    """Convert Aariz's `landmarks` list (29 entries) into our (19, 2) array.

    Looks up each of our 19 by Aariz `symbol`, then computes the two
    derived occlusal points.
    """
    aariz_by_symbol = aariz_dict(landmarks)
    out = np.full((NUM_OUR_LANDMARKS, 2), np.nan, dtype=np.float32)
    for i, name in enumerate(OUR_LANDMARK_ORDER):
        target_sym = OUR_TO_AARIZ_SYMBOL[name]
        if target_sym is None:
            continue
        if target_sym in aariz_by_symbol:
            out[i] = aariz_by_symbol[target_sym]
        else:
            raise KeyError(
                f"Aariz JSON missing symbol {target_sym!r} (needed for our "
                f"{name!r}). Symbols present: {sorted(aariz_by_symbol)}"
            )
    return derive_occlusal(out, aariz_by_symbol)


In [ ]:
%%writefile /content/ai/model.py
"""Lightweight U-Net for cephalometric landmark heat-map regression.

This is the production-quality fallback architecture for the GoldenDentist
auto-tracing pipeline.  HRNet-W32 reports ~1.5 mm mean radial error on
ISBI 2015; a U-Net of this depth typically lands around 2.5 mm — good
enough for an "AI suggestion" that the clinician verifies.

Output: a [B, K, H, W] heatmap tensor where K is the number of landmarks.
The model expects single-channel input normalised to N(0, 1).

Train with `train.py`, then export to ONNX with `export_onnx.py` and
drop the resulting `ceph.onnx` into `<project>/models/` so the browser
front-end picks it up automatically.
"""
from __future__ import annotations

import torch
import torch.nn as nn


def conv_block(in_ch: int, out_ch: int) -> nn.Sequential:
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
    )


class UNet(nn.Module):
    """A standard 4-level U-Net adapted for grayscale input."""

    def __init__(self, num_landmarks: int = 19, base: int = 32) -> None:
        super().__init__()
        self.enc1 = conv_block(1, base)
        self.enc2 = conv_block(base, base * 2)
        self.enc3 = conv_block(base * 2, base * 4)
        self.enc4 = conv_block(base * 4, base * 8)
        self.bottleneck = conv_block(base * 8, base * 16)

        self.up4 = nn.ConvTranspose2d(base * 16, base * 8, 2, stride=2)
        self.dec4 = conv_block(base * 16, base * 8)
        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.dec3 = conv_block(base * 8, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = conv_block(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = conv_block(base * 2, base)

        self.head = nn.Conv2d(base, num_landmarks, 1)
        self.pool = nn.MaxPool2d(2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        # Sigmoid keeps heat-map values in [0, 1] so the front-end can
        # interpret the peak as a confidence score directly.
        return torch.sigmoid(self.head(d1))


# Landmark order MUST match `js/ai/onnx.js::LANDMARK_ORDER`.
LANDMARK_ORDER = [
    "S", "N", "Or", "Po", "Ar",
    "ANS", "PNS", "A",
    "B", "Pog", "Gn", "Me", "Go",
    "U1", "U1A", "L1", "L1A",
    "OccA", "OccP",
]


In [ ]:
%%writefile /content/ai/dataset.py
"""Aariz / CEPHA29 dataset loader for GoldenDentist.

Verified against the Figshare release — folder layout is:

    <root>/
        train/
            Cephalograms/                        *.png / *.jpg
            Annotations/
                Cephalometric Landmarks/
                    Senior Orthodontists/        *.json
                    Junior Orthodontists/        *.json
                CVM Stages/                      *.json   (ignored)
        valid/    (same)
        test/     (same)
        cephalogram_machine_mappings.csv         per-image pixel size (mm/px)

Each image's two annotation JSONs (senior + junior) are averaged,
then mapped to our 19 landmarks via `landmarks_mapping.py`. The CSV
gives us the millimetre/pixel scale per cephalogram so MRE can be
reported in clinically meaningful units.
"""
from __future__ import annotations

import csv
import json
from pathlib import Path
from typing import Tuple

import cv2
import numpy as np
import torch
from torch.utils.data import Dataset

try:
    import albumentations as A
    HAS_ALB = True
except ImportError:
    HAS_ALB = False

from landmarks_mapping import (
    NUM_OUR_LANDMARKS,
    OUR_LANDMARK_ORDER,
    aariz_dict,
    map_aariz_to_ours,
)

# Aariz folder names are lowercase.
MODE_DIR = {"TRAIN": "train", "VALID": "valid", "TEST": "test"}


def gaussian_heatmap(h: int, w: int, x: float, y: float, sigma: float) -> np.ndarray:
    yy, xx = np.mgrid[0:h, 0:w]
    return np.exp(-((xx - x) ** 2 + (yy - y) ** 2) / (2 * sigma ** 2)).astype(np.float32)


def _read_landmarks_json(path: Path) -> list[dict]:
    """Return the raw landmark list from an Aariz JSON file."""
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data["landmarks"]


def _avg_two_raters(senior: list[dict], junior: list[dict]) -> list[dict]:
    """Average senior + junior annotations by ``symbol`` (NOT by index —
    different files can technically order their landmarks differently).

    Returns a list of dicts in the senior file's order, with each
    ``value`` replaced by the mean of the two raters.
    """
    junior_by_sym = aariz_dict(junior)
    out: list[dict] = []
    for lm in senior:
        sym = lm["symbol"]
        s_xy = np.array([lm["value"]["x"], lm["value"]["y"]], dtype=np.float32)
        j_xy = junior_by_sym.get(sym, s_xy)  # fallback to senior if missing
        m = 0.5 * (s_xy + j_xy)
        out.append({"symbol": sym, "value": {"x": float(m[0]), "y": float(m[1])}})
    return out


def load_pixel_size_table(root: Path) -> dict[str, float]:
    """Read ``cephalogram_machine_mappings.csv`` if present.
    Returns {cephalogram_id: pixel_size_mm}. Empty dict if missing.
    """
    csv_path = root / "cephalogram_machine_mappings.csv"
    if not csv_path.exists():
        return {}
    table: dict[str, float] = {}
    with open(csv_path, "r", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            try:
                table[row["cephalogram_id"]] = float(row["pixel_size"])
            except (KeyError, ValueError):
                continue
    return table


class AarizDataset(Dataset):
    """PyTorch Dataset for one split of the Aariz benchmark.

    Each sample yields ``(image_tensor, heatmaps, coords19, meta)`` where
    ``meta["pixel_size_mm"]`` is the original-resolution mm/px and
    ``meta["scale"]`` is the downsample factor applied. Multiply
    pixel-distance MRE by ``pixel_size_mm / scale`` to recover mm.

    Args:
        root:    Path to the dataset root containing train/valid/test.
        mode:    "TRAIN" | "VALID" | "TEST".
        size:    (H, W) network input size.
        sigma:   Gaussian heat-map sigma in resized-pixel units.
        augment: Apply augmentation (default: True for TRAIN only).
    """

    def __init__(
        self,
        root: str | Path,
        mode: str = "TRAIN",
        size: Tuple[int, int] = (640, 800),
        sigma: float = 3.0,
        augment: bool | None = None,
    ) -> None:
        if mode not in MODE_DIR:
            raise ValueError(f"mode must be one of {list(MODE_DIR)}, got {mode!r}")

        self.root = Path(root)
        self.mode = mode
        self.size = size
        self.sigma = sigma
        self.augment = augment if augment is not None else (mode == "TRAIN")

        split_dir = self.root / MODE_DIR[mode]
        self.images_dir = split_dir / "Cephalograms"
        self.senior_dir = split_dir / "Annotations" / "Cephalometric Landmarks" / "Senior Orthodontists"
        self.junior_dir = split_dir / "Annotations" / "Cephalometric Landmarks" / "Junior Orthodontists"

        if not self.images_dir.exists():
            raise FileNotFoundError(f"Missing images directory: {self.images_dir}")
        if not (self.senior_dir.exists() and self.junior_dir.exists()):
            raise FileNotFoundError(
                f"Missing annotations under {split_dir / 'Annotations'}"
            )

        self.images = sorted(p for p in self.images_dir.iterdir() if p.is_file())
        if not self.images:
            raise FileNotFoundError(f"No images in {self.images_dir}")

        self.pixel_size_mm = load_pixel_size_table(self.root)
        if not self.pixel_size_mm:
            print(
                "[dataset] cephalogram_machine_mappings.csv not found — "
                "MRE will be reported in pixels only."
            )

        self.aug = None
        if self.augment and HAS_ALB:
            self.aug = A.Compose([
                A.RandomBrightnessContrast(p=0.5),
                A.GaussNoise(p=0.3),
                A.ShiftScaleRotate(
                    shift_limit=0.05, scale_limit=0.05, rotate_limit=5,
                    p=0.5, border_mode=0,
                ),
                # No HorizontalFlip — flipping reverses anatomy.
            ], keypoint_params=A.KeypointParams(format="xy", remove_invisible=False))
        elif self.augment and not HAS_ALB:
            print("[dataset] albumentations not installed — training without augmentation")

    # ---- API --------------------------------------------------------

    def __len__(self) -> int:
        return len(self.images)

    def __getitem__(self, idx: int):
        img_path = self.images[idx]
        stem = img_path.stem
        senior_path = self.senior_dir / f"{stem}.json"
        junior_path = self.junior_dir / f"{stem}.json"

        # 1. Image (read as grayscale).
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise IOError(f"Failed to read {img_path}")
        h0, w0 = img.shape

        # 2. Annotations: senior + junior averaged by symbol.
        senior = _read_landmarks_json(senior_path)
        junior = _read_landmarks_json(junior_path)
        averaged = _avg_two_raters(senior, junior)

        # 3. Map to our 19 landmarks.
        coords19 = map_aariz_to_ours(averaged)

        # 4. Resize image and rescale coordinates.
        target_h, target_w = self.size
        img = cv2.resize(img, (target_w, target_h), interpolation=cv2.INTER_AREA)
        sx = target_w / w0
        sy = target_h / h0
        coords19[:, 0] *= sx
        coords19[:, 1] *= sy

        # 5. Augmentation (training only).
        np_img = img.astype(np.float32) / 255.0
        if self.aug is not None:
            kp = [(float(x), float(y)) for x, y in coords19]
            out = self.aug(image=np_img, keypoints=kp)
            np_img = out["image"]
            coords19 = np.asarray(out["keypoints"], dtype=np.float32)
            if coords19.shape[0] != NUM_OUR_LANDMARKS:
                pad = np.full(
                    (NUM_OUR_LANDMARKS - coords19.shape[0], 2), -1.0, dtype=np.float32
                )
                coords19 = np.concatenate([coords19, pad], axis=0)[:NUM_OUR_LANDMARKS]

        # 6. Normalize.
        np_img = (np_img - 0.5) / 0.5
        np_img = np_img[None, ...]  # add channel

        # 7. Heatmaps.
        heatmaps = np.zeros((NUM_OUR_LANDMARKS, target_h, target_w), dtype=np.float32)
        for k, (x, y) in enumerate(coords19):
            if 0 <= x < target_w and 0 <= y < target_h and not np.isnan(x):
                heatmaps[k] = gaussian_heatmap(target_h, target_w, x, y, self.sigma)

        # 8. Per-image scale info so train.py can convert pixel-MRE → mm.
        ps_mm = self.pixel_size_mm.get(stem, 0.0)  # original mm per native pixel
        # In the network's resized pixel space, one pixel covers
        # (ps_mm / mean(sx, sy)) mm of physical distance (approx).
        scale_mean = 0.5 * (sx + sy)
        mm_per_resized_px = ps_mm / scale_mean if (ps_mm and scale_mean) else 0.0

        meta = {
            "stem": stem,
            "scale_x": float(sx),
            "scale_y": float(sy),
            "orig_size": (int(w0), int(h0)),
            "pixel_size_mm": float(ps_mm),
            "mm_per_resized_px": float(mm_per_resized_px),
        }
        return (
            torch.from_numpy(np_img),
            torch.from_numpy(heatmaps),
            torch.from_numpy(coords19.astype(np.float32)),
            meta,
        )


In [ ]:
%%writefile /content/ai/train.py
"""Train the GoldenDentist U-Net on the Aariz / CEPHA29 dataset.

Usage:
    python ai/train.py --data path/to/Aariz --epochs 80 --batch 4

Aariz ships with pre-defined train/valid/test splits; this script
honors them. Validation MRE is reported in millimetres (using each
image's pixel size from cephalogram_machine_mappings.csv) so the
result is directly comparable to the published Aariz benchmarks. Best
checkpoint by val MRE is saved as ``ai/checkpoints/best.pt``. Run
``ai/export_onnx.py`` afterwards.
"""
from __future__ import annotations

import sys
# Force UTF-8 stdout/stderr so non-ASCII characters print on Windows
# consoles whose default code page is cp1256 / cp1252.
try:
    sys.stdout.reconfigure(encoding="utf-8")
    sys.stderr.reconfigure(encoding="utf-8")
except Exception:
    pass

import argparse
import math
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

from dataset import AarizDataset
from landmarks_mapping import OUR_LANDMARK_ORDER
from model import UNet


def heatmap_to_coords(hm: torch.Tensor) -> torch.Tensor:
    """Argmax + sub-pixel parabolic refinement on a [B, K, H, W] heatmap.
    Returns [B, K, 2] (x, y)."""
    b, k, h, w = hm.shape
    flat = hm.view(b, k, -1)
    _, idx = flat.max(dim=-1)
    ys = (idx // w).float()
    xs = (idx % w).float()
    for bi in range(b):
        for ki in range(k):
            cx, cy = int(xs[bi, ki].item()), int(ys[bi, ki].item())
            if 0 < cx < w - 1 and 0 < cy < h - 1:
                c = hm[bi, ki, cy, cx].item()
                xm = hm[bi, ki, cy, cx - 1].item()
                xp = hm[bi, ki, cy, cx + 1].item()
                ym = hm[bi, ki, cy - 1, cx].item()
                yp = hm[bi, ki, cy + 1, cx].item()
                dx = (xm - xp) / (2 * (xm - 2 * c + xp + 1e-6))
                dy = (ym - yp) / (2 * (ym - 2 * c + yp + 1e-6))
                xs[bi, ki] += max(-1.0, min(1.0, dx))
                ys[bi, ki] += max(-1.0, min(1.0, dy))
    return torch.stack([xs, ys], dim=-1)


def per_landmark_radial_errors(
    pred: torch.Tensor, gt: torch.Tensor, mm_per_px: torch.Tensor | None = None,
) -> np.ndarray:
    """Per-landmark radial errors for a single batch.

    Returns array of shape [B, K] in mm if `mm_per_px` is provided,
    otherwise in pixels.
    """
    diffs = (pred - gt).pow(2).sum(-1).sqrt()  # [B, K] in resized pixels
    if mm_per_px is not None:
        diffs = diffs * mm_per_px[:, None]
    return diffs.numpy()


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True, help="Aariz dataset root containing train/valid/test")
    ap.add_argument("--epochs", type=int, default=80)
    ap.add_argument("--batch", type=int, default=4)
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--size", type=int, nargs=2, default=[640, 800])
    ap.add_argument("--sigma", type=float, default=3.0)
    ap.add_argument("--ckpt-dir", default=str(Path(__file__).parent / "checkpoints"))
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument(
        "--amp", action="store_true",
        help="Enable mixed-precision (fp16) training. Saves VRAM and "
        "is ~30%% faster on Ampere/Ada GPUs. Recommended on RTX 30/40 series.",
    )
    ap.add_argument(
        "--pos-weight", type=float, default=200.0,
        help="Multiplier for positive (peak) pixels in the heatmap MSE loss. "
        "0 = vanilla MSE (collapses to all-zero predictions due to extreme "
        "class imbalance). 100-300 typically works well. Default: 200.",
    )
    args = ap.parse_args()

    Path(args.ckpt_dir).mkdir(parents=True, exist_ok=True)
    print(f"Device: {args.device}")

    train_ds = AarizDataset(args.data, mode="TRAIN", size=tuple(args.size), sigma=args.sigma, augment=True)
    val_ds   = AarizDataset(args.data, mode="VALID", size=tuple(args.size), sigma=args.sigma, augment=False)
    print(f"Train: {len(train_ds)}  |  Valid: {len(val_ds)}")

    train_dl = DataLoader(train_ds, batch_size=args.batch, shuffle=True,  num_workers=args.workers, pin_memory=True)
    val_dl   = DataLoader(val_ds,   batch_size=args.batch, shuffle=False, num_workers=args.workers, pin_memory=True)

    model = UNet(num_landmarks=len(OUR_LANDMARK_ORDER)).to(args.device)
    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=args.epochs)

    # Weighted MSE for heat-map regression. With Gaussian sigma=3 only ~0.1%
    # of pixels carry useful signal, so plain MSE collapses to "predict zero
    # everywhere". Weighting positive pixels by `pos_weight` makes them
    # account for ~50% of total loss mass, which forces the network to
    # actually localize the peaks.
    pos_weight = float(args.pos_weight)

    def loss_fn(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        weight = 1.0 + pos_weight * target  # background=1, peak=1+pos_weight
        return ((pred - target).pow(2) * weight).mean()

    use_amp = bool(args.amp) and args.device.startswith("cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    if use_amp:
        print("Mixed-precision (AMP) training: ENABLED")

    best = math.inf
    for epoch in range(args.epochs):
        # ---- train ----------------------------------------------------
        model.train()
        running = 0.0
        pbar = tqdm(train_dl, desc=f"epoch {epoch+1}/{args.epochs}")
        for img, hm, _, _ in pbar:
            img = img.to(args.device, non_blocking=True)
            hm = hm.to(args.device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=use_amp):
                pred = model(img)
                loss = loss_fn(pred, hm)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            running += loss.item() * img.size(0)
            pbar.set_postfix(loss=loss.item())
        sched.step()
        train_loss = running / len(train_ds)

        # ---- validation ----------------------------------------------
        model.eval()
        all_errs: list[np.ndarray] = []
        with torch.no_grad():
            for img, _, coords, meta in val_dl:
                img = img.to(args.device)
                with torch.amp.autocast("cuda", enabled=use_amp):
                    pred = model(img).float().cpu()
                pred_xy = heatmap_to_coords(pred)
                mm_per_px = torch.as_tensor(
                    meta["mm_per_resized_px"], dtype=torch.float32
                )
                if (mm_per_px > 0).all():
                    errs = per_landmark_radial_errors(pred_xy, coords, mm_per_px)
                    unit = "mm"
                else:
                    errs = per_landmark_radial_errors(pred_xy, coords, None)
                    unit = "px"
                all_errs.append(errs)

        all_errs_cat = np.concatenate(all_errs, axis=0)  # [N, K]
        per_lm = np.nanmean(all_errs_cat, axis=0)        # [K]
        mre = float(np.nanmean(per_lm))

        # SDR — Success Detection Rate at clinical thresholds (mm only).
        if unit == "mm":
            sdr_thresholds = (2.0, 2.5, 3.0, 4.0)
            sdr = {t: float(np.mean(all_errs_cat <= t) * 100) for t in sdr_thresholds}
        else:
            sdr = {}

        print(
            f"epoch {epoch+1}: train_loss={train_loss:.4f}  "
            f"val MRE = {mre:.3f} {unit}"
            + (
                "  SDR%@(2,2.5,3,4)mm = "
                + ", ".join(f"{sdr[t]:.1f}" for t in (2.0, 2.5, 3.0, 4.0))
                if sdr else ""
            )
        )
        for name, e in zip(OUR_LANDMARK_ORDER, per_lm):
            print(f"    {name:5s} {e:6.3f} {unit}")

        ckpt = Path(args.ckpt_dir) / f"epoch_{epoch+1:03d}.pt"
        torch.save({"model": model.state_dict(), "mre": mre, "unit": unit, "epoch": epoch + 1}, ckpt)
        if mre < best:
            best = mre
            torch.save(
                {"model": model.state_dict(), "mre": mre, "unit": unit, "epoch": epoch + 1},
                Path(args.ckpt_dir) / "best.pt",
            )
            print(f"  -> new best MRE {best:.3f} {unit} (saved best.pt)")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/ai/export_onnx.py
"""Export a trained U-Net checkpoint to a single-file ONNX model for
in-browser inference via onnxruntime-web.

Usage:
    python ai/export_onnx.py --ckpt ai/checkpoints/best.pt --out models/ceph.onnx
"""
from __future__ import annotations

import sys
try:
    sys.stdout.reconfigure(encoding="utf-8")
    sys.stderr.reconfigure(encoding="utf-8")
except Exception:
    pass

import argparse
from pathlib import Path

import onnx
import torch

from model import UNet


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--ckpt", required=True)
    ap.add_argument("--out", default="models/ceph.onnx")
    ap.add_argument("--landmarks", type=int, default=19)
    ap.add_argument("--size", type=int, nargs=2, default=[640, 800])  # H W (must match front-end)
    ap.add_argument("--opset", type=int, default=17)
    args = ap.parse_args()

    out_path = Path(args.out)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    model = UNet(num_landmarks=args.landmarks)
    state = torch.load(args.ckpt, map_location="cpu", weights_only=False)
    model.load_state_dict(state["model"] if "model" in state else state)
    model.eval()

    # Use the legacy (non-dynamo) torch.onnx exporter. The dynamo-based one
    # in PyTorch 2.11 splits weights into a sidecar .data file by default,
    # which onnxruntime-web cannot follow when fetching a single URL.
    dummy = torch.zeros(1, 1, args.size[0], args.size[1])
    torch.onnx.export(
        model, dummy, str(out_path),
        input_names=["input"], output_names=["heatmap"],
        opset_version=args.opset,
        dynamic_axes={"input": {0: "batch"}, "heatmap": {0: "batch"}},
        dynamo=False,
    )

    # Belt-and-braces: if any external-data file was somehow produced, fold
    # it into the main model file and delete the sidecar so the front-end
    # only has to fetch one URL.
    sidecar = out_path.with_suffix(out_path.suffix + ".data")
    m = onnx.load(str(out_path), load_external_data=True)
    onnx.save(m, str(out_path), save_as_external_data=False)
    if sidecar.exists():
        sidecar.unlink()
        print(f"  (folded sidecar weights from {sidecar.name})")

    size_mb = out_path.stat().st_size / 1024**2
    onnx.checker.check_model(m)
    print(f"Exported {out_path}  ({size_mb:.1f} MB, single file)")
    if size_mb < 5:
        print("  WARNING: file is suspiciously small. Verify it contains weights.")
    print("Drop this file at  <project>/models/ceph.onnx  and the front-end will pick it up.")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/ai/verify_dataset.py
"""Visualize Aariz samples to confirm the landmark mapping is correct
BEFORE you spend hours training.

Each output PNG shows:
    * All 29 raw Aariz landmarks labelled with their `symbol` (green).
    * Our 19 mapped landmarks labelled with their name (gold).
    * Derived landmarks (OccA) in pink. OccP is also pink IF UMT/LMT
      were absent and we had to fall back to the geometric estimate;
      otherwise OccP is gold (a real molar midpoint).

Open the resulting PNGs in ``ai/verify_previews/``. If a gold-labelled
dot lands on the wrong anatomy, edit ``OUR_TO_AARIZ_SYMBOL`` in
``landmarks_mapping.py``.

Usage:
    python ai/verify_dataset.py --data path/to/Aariz --n 5 --split TRAIN
"""
from __future__ import annotations

import sys
try:
    sys.stdout.reconfigure(encoding="utf-8")
    sys.stderr.reconfigure(encoding="utf-8")
except Exception:
    pass

import argparse
import json
from pathlib import Path

import cv2
import numpy as np

from landmarks_mapping import (
    OUR_LANDMARK_ORDER,
    OUR_TO_AARIZ_SYMBOL,
    aariz_dict,
    map_aariz_to_ours,
)

MODE_DIR = {"TRAIN": "train", "VALID": "valid", "TEST": "test"}


def _avg_landmarks(senior_json: Path, junior_json: Path) -> list[dict]:
    """Average senior + junior annotations by symbol (matches dataset.py)."""
    with open(senior_json, "r", encoding="utf-8") as f:
        senior = json.load(f)["landmarks"]
    with open(junior_json, "r", encoding="utf-8") as f:
        junior = json.load(f)["landmarks"]
    j_by_sym = aariz_dict(junior)
    out: list[dict] = []
    for lm in senior:
        sym = lm["symbol"]
        sx, sy = lm["value"]["x"], lm["value"]["y"]
        if sym in j_by_sym:
            jx, jy = j_by_sym[sym]
            mx, my = 0.5 * (sx + jx), 0.5 * (sy + jy)
        else:
            mx, my = sx, sy
        out.append({"symbol": sym, "value": {"x": float(mx), "y": float(my)}})
    return out


def draw_preview(image_path: Path, senior_json: Path, junior_json: Path, out_path: Path) -> None:
    img = cv2.imread(str(image_path))
    if img is None:
        raise IOError(f"Cannot read {image_path}")

    averaged = _avg_landmarks(senior_json, junior_json)
    by_sym = aariz_dict(averaged)
    coords19 = map_aariz_to_ours(averaged)

    has_molars = ("UMT" in by_sym) and ("LMT" in by_sym)

    # Draw all 29 raw Aariz landmarks (green, with symbol).
    for sym, (x, y) in by_sym.items():
        cv2.circle(img, (int(x), int(y)), 6, (0, 220, 0), -1)
        cv2.putText(
            img, sym, (int(x) + 8, int(y) - 8),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 220, 0), 2,
        )

    # Draw our 19 mapped landmarks (gold for direct, pink for derived/fallback).
    for i, name in enumerate(OUR_LANDMARK_ORDER):
        x, y = coords19[i]
        if np.isnan(x):
            continue
        is_direct = OUR_TO_AARIZ_SYMBOL.get(name) is not None
        if name == "OccP":
            is_direct = has_molars  # pink only if we had to fall back
        color = (0, 215, 215) if is_direct else (180, 105, 255)
        cv2.circle(img, (int(x), int(y)), 11, color, 2)
        cv2.putText(
            img, name, (int(x) + 14, int(y) + 6),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2,
        )

    cv2.imwrite(str(out_path), img)


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True, help="Aariz dataset root")
    ap.add_argument("--split", default="TRAIN", choices=["TRAIN", "VALID", "TEST"])
    ap.add_argument("--n", type=int, default=5, help="Number of samples to render")
    ap.add_argument("--out", default=str(Path(__file__).parent / "verify_previews"))
    args = ap.parse_args()

    root = Path(args.data) / MODE_DIR[args.split]
    images_dir = root / "Cephalograms"
    senior_dir = root / "Annotations" / "Cephalometric Landmarks" / "Senior Orthodontists"
    junior_dir = root / "Annotations" / "Cephalometric Landmarks" / "Junior Orthodontists"

    images = sorted(p for p in images_dir.iterdir() if p.is_file())
    if not images:
        raise SystemExit(f"No images in {images_dir}")

    out_dir = Path(args.out)
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"\nLandmark mapping (our 19 <-- Aariz):")
    for name in OUR_LANDMARK_ORDER:
        sym = OUR_TO_AARIZ_SYMBOL[name]
        print(f"  {name:5s} <-- {sym if sym else '(derived)'}")

    print(f"\nRendering {min(args.n, len(images))} previews into {out_dir} ...")
    for img_path in images[: args.n]:
        stem = img_path.stem
        senior_p = senior_dir / f"{stem}.json"
        junior_p = junior_dir / f"{stem}.json"
        if not (senior_p.exists() and junior_p.exists()):
            print(f"  skip {stem}: missing annotation")
            continue
        out_p = out_dir / f"{stem}_preview.png"
        draw_preview(img_path, senior_p, junior_p, out_p)
        print(f"  wrote {out_p.name}")

    print(
        "\nOpen the previews. Each green-labelled dot is a raw Aariz landmark "
        "(with its symbol). Each gold/pink-labelled dot is one of our 19. "
        "Pink = derived (OccA always; OccP only if UMT/LMT missing).\n\n"
        "If a labelled dot lands on the WRONG anatomical feature, edit "
        "OUR_TO_AARIZ_SYMBOL in landmarks_mapping.py and re-run this script."
    )


if __name__ == "__main__":
    main()


## 4. Mount Google Drive and point at the dataset

Upload your `Aariz` folder (containing `train/`, `valid/`, `test/`, and `cephalogram_machine_mappings.csv`) into your Drive once. Then set `DATA_DIR` below to its path.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ▼ EDIT THIS LINE: path inside your Drive ▼
DATA_DIR = '/content/drive/MyDrive/Aariz'

import pathlib
for sp in ['train', 'valid', 'test']:
    p = pathlib.Path(DATA_DIR) / sp / 'Cephalograms'
    n = len(list(p.glob('*'))) if p.exists() else 0
    flag = 'OK ' if n else 'MISSING'
    print(f'  [{flag}] {sp}: {n} images at {p}')
csv_p = pathlib.Path(DATA_DIR) / 'cephalogram_machine_mappings.csv'
print(f'  [{ "OK " if csv_p.exists() else "MISSING" }] CSV: {csv_p}')


## 5. Verify the landmark mapping (recommended)

Renders 5 train images with all 29 raw Aariz landmarks (green) and our 19 mapped landmarks (gold / pink). Open the previews in the file browser and confirm anatomy is correct before training.

In [ ]:
%cd /content/ai
!python verify_dataset.py --data "$DATA_DIR" --split TRAIN --n 5

In [ ]:
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('/content/ai/verify_previews/*.png'))[:2]:
    print(p)
    display(Image(p, width=600))


## 6. Train

Each epoch prints validation MRE in **millimetres** plus SDR % at 2/2.5/3/4 mm — directly comparable to the published Aariz benchmark. Best checkpoint by validation MRE is kept at `ai/checkpoints/best.pt`. Roughly 1–1.5 hours on a T4 for 80 epochs.

In [ ]:
%cd /content/ai
!python train.py --data "$DATA_DIR" --epochs 80 --batch 4

## 7. Export to ONNX and download to your machine

In [ ]:
%cd /content/ai
!python export_onnx.py --ckpt /content/ai/checkpoints/best.pt --out /content/ai/ceph.onnx
from google.colab import files
files.download('/content/ai/ceph.onnx')


## 8. Drop into the project

Save the downloaded `ceph.onnx` at
`<project>/models/ceph.onnx` on your machine. Refresh the GoldenDentist page in your browser. The toolbar readout switches from **AI: heuristic atlas** to **AI: ONNX trained model** and the **Auto-Trace (AI)** button now uses your trained model.